# VPA Demo: LLM Agent Planning And Execution

Use this notebook when you want to describe the scientific goal in
natural language and let the VPA agent choose tools or workflows. The
request below asks for a design loop centered on deep-learning tools:
RFdiffusion, ProteinMPNN, Protenix-v2, QC, and ranking.

By default, this notebook is smoke-safe: it checks GPU/CLI visibility and
prints agent commands without calling an LLM. Set `VPA_DEMO_RUN_LLM=1`
and configure a provider to execute plan-only calls. Add
`VPA_DEMO_RUN_HEAVY=1` only when you also want the agent to launch and
wait for model-backed workflow execution.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import textwrap
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "vibeproteinagent").exists():
            return candidate
    return Path("/opt/vpa-data/VibeProteinAgent")

REPO_ROOT = find_repo_root()
VPA = os.environ.get("VPA_BIN", str(Path("/opt/miniconda3/envs/vpa/bin/vpa")))
if not Path(VPA).exists():
    VPA = os.environ.get("VPA_BIN", "vpa")

RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", "demo")

print(f"repo: {REPO_ROOT}")
print(f"vpa: {VPA}")
print(f"RUN_INSTALL={RUN_INSTALL} RUN_HEAVY={RUN_HEAVY} RUN_LLM={RUN_LLM}")


def run(cmd, *, check=True, cwd=REPO_ROOT, env=None, timeout=None):
    if isinstance(cmd, str):
        printable = cmd
        shell = True
    else:
        printable = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {printable}")
    proc = subprocess.run(
        cmd,
        shell=shell,
        cwd=str(cwd),
        env={**os.environ, **(env or {})},
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout[-6000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {printable}")
    return proc


def show_command(parts):
    print(" ".join(shlex.quote(str(part)) for part in parts))


## Optional Install And Provider Credentials

Run this cell only when the local VPA install is missing or you want the notebook to bootstrap the agent profile. The agent profile itself does not need an OpenRouter key; `OPENROUTER_API_KEY` is only needed when `VPA_DEMO_PROVIDER=openrouter` and `VPA_DEMO_RUN_LLM=1`.

For local CLI providers, log in once in the same user account that will run this notebook or the Slurm job: `codex login` for Codex, and `claude /login` for Claude Code. These are local CLI sessions, not values to paste into the repository.


In [ ]:
install_cmd = ["npx", "vibeproteinagent", "install", "--profile", "agent", "--cuda-profile", "auto", "--yes"]
if RUN_INSTALL:
    run(install_cmd, timeout=3600)
else:
    show_command(install_cmd)

provider = os.environ.get("VPA_DEMO_PROVIDER", "codex")
if provider == "openrouter" and RUN_LLM and not (
    os.environ.get("OPENROUTER_API_KEY") or os.environ.get("VPA_OPENROUTER_CREDENTIAL")
):
    raise RuntimeError(
        "Set OPENROUTER_API_KEY=<openrouter-api-key> before running OpenRouter-backed agent calls."
    )

print("Provider setup commands and credential placeholders, not real values:")
show_command(["codex", "login"])
show_command(["claude", "/login"])
show_command(["bash", "-lc", "export OPENROUTER_API_KEY=<openrouter-api-key>"])


## 1. GPU, Public Tool Surface, And Workflow Templates

In [ ]:
# This cell is intentionally lightweight. It proves the notebook is running on a
# GPU-capable node and that PyTorch can see CUDA. The heavy VPA jobs below are
# gated behind RUN_HEAVY so the tutorial remains safe to execute.
run("hostname")
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -3", check=False)
import sys
run([sys.executable, "-c", "import torch; print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('device0', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"])

In [ ]:
run([VPA, "tools", "--json"], check=False, timeout=90)
run([VPA, "workflow", "list"], check=False, timeout=90)

## 2. Plan-Only Agent Request

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_agent"
request = "Design binder backbones, inverse-fold them, predict/refold top candidates, run QC, and rank the best design. Prefer local GPU tools and avoid remote APIs."
plan_cmd = [
    VPA, "agent", "run", request,
    "--provider", os.environ.get("VPA_DEMO_PROVIDER", "codex"),
    "--model", os.environ.get("VPA_DEMO_MODEL", "gpt-5.2"),
    "--workspace", workspace,
    "--plan-only",
    "--json",
]
if RUN_LLM:
    run(plan_cmd, timeout=900)
else:
    show_command(plan_cmd)

## 3. Execute In One Command

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_agent"
execute_cmd = [
    VPA, "agent", "run", request,
    "--provider", os.environ.get("VPA_DEMO_PROVIDER", "codex"),
    "--model", os.environ.get("VPA_DEMO_MODEL", "gpt-5.2"),
    "--workspace", workspace,
    "--max-rounds", "8",
    "--max-entries", "3",
    "--max-wait-s", "7200",
    "--json",
]
if RUN_LLM and RUN_HEAVY:
    run([VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "2", "--workspace", workspace], timeout=120)
    run(execute_cmd, timeout=7800)
else:
    show_command([VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "2", "--workspace", workspace])
    show_command(execute_cmd)

## 4. Inspect Agent History And Human Reports

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_agent"
for parts in [
    [VPA, "agent", "runs", "--workspace", workspace],
    [VPA, "agent", "show", "<agent_run_id>", "--workspace", workspace],
    [VPA, "workflow", "status", "<workflow_id>", "--workspace", workspace],
    [VPA, "report", "<workflow_id>", "--workspace", workspace, "--open"],
]:
    show_command(parts)

## 5. Provider Notes

In [ ]:
print("Provider options:")
print("  codex       -> local Codex CLI login/session; run `codex login` first")
print("  claude-code -> local Claude Code CLI login/session; run `claude /login` first")
print("  openrouter  -> OPENROUTER_API_KEY plus an OpenRouter model id")
print()
print("Examples:")
show_command([VPA, "agent", "run", request, "--provider", "claude-code", "--model", "sonnet", "--plan-only"])
show_command([VPA, "agent", "run", request, "--provider", "openrouter", "--model", "openrouter/openai/gpt-5.1", "--plan-only"])
